# 🎖️ 2026년 부대 맞춤형 AI·SW 소양교육
## 육군 직무보수교육 - **4일차 미니 프로젝트**

> **Ⅳ. 부대 실무 데이터를 활용한 EDA 및 시각화**

### 🎯 프로젝트 주제
> **"부대 장비 정비 기록 분석을 통한 예방정비 전략 수립"**

### 📚 오늘 배울 내용
- **EDA(탐색적 데이터 분석)** 개념 및 6단계 프로세스
- 부대 장비 정비 데이터 로드 & 품질 점검
- 기술통계 분석 (describe, value_counts)
- 단변량 분석 (히스토그램, 박스플롯)
- 다변량 분석 (groupby, corr, heatmap)
- 차트 선택 가이드 (6가지 차트 유형)
- **인사이트 도출 및 보고서 작성**

---

### 💡 John Tukey의 EDA 정의 (1977)
> **"데이터가 우리에게 무엇을 말하는지 먼저 들어라."**  
> 데이터를 분석하기 전, 그래프·통계로 **구조·패턴·이상치·관계**를 직관적으로 파악하는 탐색 과정

### 🏛 부대 현장 활용 사례
- **🔧 장비 정비**: 고장 빈도·정비소요시간 분석 → 예방정비 시기 예측
- **🎯 훈련 성과**: 사격·체력검정 결과 분포 및 추세 파악
- **📦 군수 현황**: 물자 소비 패턴 분석 → 적정 재고 수준 도출

---

# 🛠️ Chapter 0. 환경 구성 및 실습 데이터 준비

## 0-1. 라이브러리 import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f"pandas    : {pd.__version__}")
print(f"numpy     : {np.__version__}")
print(f"seaborn   : {sns.__version__}")

## 0-2. 한글 폰트 설정

In [ ]:
!apt-get install -qq fonts-nanum > /dev/null 2>&1

import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
sns.set_style('whitegrid')
sns.set_context('notebook')
print("✅ 한글 폰트 설정 완료")

## 0-3. 실습 데이터 생성 — 부대 장비 정비 기록

PPT에서 제시된 시나리오(240건 × 10컬럼 × 12개월)를 그대로 재현합니다.

**데이터 명세**
- 기간: 2024년 1월 ~ 2024년 12월 (12개월)
- 장비유형 4종: K131 트럭, K200 장갑차, 발전기, K511 카고트럭
- 고장유형 5종: 엔진 이상, 변속기 결함, 브레이크 마모, 궤도 마모, 오일 누유
- 의도적 삽입:
  - 결측값 (정비일자 2건, 고장유형 5건)
  - 이상치 (정비시간 38.5h 초과)
  - 계절성 (8월 훈련 후 급증)

In [ ]:
np.random.seed(42)
N = 240

# 장비유형별 특성 (정비시간 평균, 비용 평균)
equip_specs = {
    "K131 트럭":   {"time_mean": 6.0,  "cost_mean": 100, "weight": 0.30, "code": "K131"},
    "K200 장갑차": {"time_mean": 18.0, "cost_mean": 480, "weight": 0.25, "code": "K200"},
    "K511 카고":   {"time_mean": 8.0,  "cost_mean": 150, "weight": 0.25, "code": "K511"},
    "발전기":      {"time_mean": 4.0,  "cost_mean": 85,  "weight": 0.20, "code": "GEN"},
}

equip_types = list(equip_specs.keys())
weights     = [equip_specs[e]["weight"] for e in equip_types]

breakdown_types = ["엔진 이상", "변속기 결함", "브레이크 마모", "궤도 마모", "오일 누유"]
units   = ["1중대", "2중대", "3중대", "4중대"]
results = ["완료", "부분완료", "가동불능"]

# 날짜 생성 - 8월에 정비 건수 몰리도록 (하계 훈련 후)
month_weights = [0.06, 0.06, 0.07, 0.08, 0.08, 0.09, 0.09, 0.15, 0.10, 0.08, 0.07, 0.07]
month_weights = np.array(month_weights) / sum(month_weights)
months = np.random.choice(range(1, 13), size=N, p=month_weights)

dates = []
for m in months:
    d = np.random.randint(1, 29)
    dates.append(f"2024-{m:02d}-{d:02d}")

# 장비유형 배정
types = np.random.choice(equip_types, N, p=weights)

# 장비번호: 코드-번호 (K131-003 형식)
equip_ids = [f"{equip_specs[t]['code']}-{np.random.randint(1, 30):03d}" for t in types]

# 정비시간: 장비유형에 따라 다른 평균, 일부 이상치
times = []
for t in types:
    base = equip_specs[t]["time_mean"]
    # 5% 확률로 이상치 (아주 긴 정비)
    if np.random.random() < 0.05:
        times.append(np.round(np.random.uniform(38, 48), 1))
    else:
        times.append(np.round(max(0.5, np.random.gamma(2.5, base/2.5)), 1))

# 정비비용: 정비시간과 강한 양의 상관 + 장비별 가중치
costs = []
for t, time in zip(types, times):
    cost_base = equip_specs[t]["cost_mean"]
    cost = cost_base * (time / equip_specs[t]["time_mean"]) * np.random.uniform(0.7, 1.3)
    costs.append(int(max(12, cost)))

# 고장유형 - 장비별 특성 반영
breakdowns = []
for t in types:
    if t == "K200 장갑차":
        bd = np.random.choice(["변속기 결함", "궤도 마모", "엔진 이상"], p=[0.4, 0.4, 0.2])
    elif t == "발전기":
        bd = np.random.choice(["엔진 이상", "오일 누유"], p=[0.6, 0.4])
    else:
        bd = np.random.choice(breakdown_types)
    breakdowns.append(bd)

# 정비 결과
result_list = np.random.choice(results, N, p=[0.80, 0.12, 0.08])

# DataFrame 생성
df_raw = pd.DataFrame({
    "정비일자": dates,
    "장비번호": equip_ids,
    "장비유형": types,
    "고장유형": breakdowns,
    "담당부대": np.random.choice(units, N),
    "정비시간(h)": times,
    "정비비용(만원)": costs,
    "결과": result_list,
})

# 의도적으로 결측값 삽입
df_raw.loc[[13, 87], "정비일자"] = np.nan                          # 2건
df_raw.loc[[5, 40, 100, 150, 200], "고장유형"] = np.nan            # 5건

# 저장
df_raw.to_csv('/content/maintenance_log.csv', index=False, encoding='utf-8-sig')

print(f"✅ 정비 기록 생성: {df_raw.shape[0]}건 × {df_raw.shape[1]}컬럼")
print(f"   기간        : 2024년 1월 ~ 12월")
print(f"   장비유형    : {df_raw['장비유형'].nunique()}종")
print(f"   결측값      : 정비일자 2건, 고장유형 5건 (의도적 삽입)")
print(f"   저장 위치   : /content/maintenance_log.csv")
df_raw.head()

---
# 📚 Chapter 1. EDA 개념과 6단계 프로세스

## 1-1. EDA란 무엇인가?

### 존 튜키(John Tukey)의 정의 — 1977
> 데이터를 분석하기 전, **그래프·통계 방법**으로  
> 데이터의 **구조·패턴·이상치·관계**를 직관적으로 파악하는 **탐색 과정**

### EDA가 중요한 4가지 이유

| 목적 | 설명 |
|---|---|
| 🧐 **품질 문제 발견** | 결측값·오류·이상치 사전 탐지 |
| 🎯 **분석 방향 설정** | 모델링 전 적합한 전략 수립 |
| 💡 **패턴·인사이트** | 예상치 못한 발견 |
| 📣 **시각적 전달** | 비전문가에게도 쉽게 설명 |

## 1-2. EDA 6단계 프로세스

```
┌─────────────────────────────────────────────────────┐
│ ① 데이터 로드      │ df.info() / df.shape           │
│ ② 품질 점검        │ isnull() / duplicated()        │
│ ③ 기술통계 분석    │ describe() / value_counts()    │
│ ④ 단변량 분석      │ hist() / boxplot()             │
│ ⑤ 다변량 분석      │ corr() / groupby() / heatmap   │
│ ⑥ 인사이트 도출    │ 의사결정 권고안 작성            │
└─────────────────────────────────────────────────────┘
```

오늘은 이 6단계를 **부대 장비 정비 데이터** 로 직접 실행해 봅니다.

---
### 🔥 실습문제 1 - EDA 개념 확인
다음 질문에 답하세요.

1. EDA라는 개념을 처음 제시한 통계학자의 이름은?
2. EDA 6단계 중 `describe()`가 사용되는 단계는 몇 단계?
3. 아래 상황에 적합한 EDA 단계를 매칭하세요.
   - (a) "정비시간 평균과 표준편차를 알고 싶다"
   - (b) "정비시간과 정비비용이 관련 있을까?"
   - (c) "결측값이 몇 건 있는지 확인"


In [ ]:
# ✍️ 여기에 답을 작성하세요

answer_1 = ""        # 이름
answer_2 = 0         # 숫자
match_a = 0          # 1~6 중 해당 단계
match_b = 0
match_c = 0

print(f"1) EDA 창시자: {answer_1}")
print(f"2) describe() 단계: {answer_2}")
print(f"3-a) 평균·표준편차 분석 → {match_a}단계")
print(f"3-b) 변수 간 관련성 → {match_b}단계")
print(f"3-c) 결측값 확인 → {match_c}단계")

**✅ 정답**

In [ ]:
answer_1 = "John Tukey (존 튜키)"
answer_2 = 3
match_a  = 3       # 기술통계 분석
match_b  = 5       # 다변량 분석
match_c  = 2       # 품질 점검

print(f"1) EDA 창시자: {answer_1}")
print(f"2) describe() 단계: {answer_2}단계 (기술통계 분석)")
print(f"3-a) 평균·표준편차 → {match_a}단계 (기술통계 분석)")
print(f"3-b) 변수 간 관련성 → {match_b}단계 (다변량 분석)")
print(f"3-c) 결측값 확인 → {match_c}단계 (품질 점검)")

---
# 📥 Chapter 2. [Step 1-2] 데이터 로드 & 품질 점검

## 2-1. 데이터 로드

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('/content/maintenance_log.csv', encoding='utf-8-sig')

# 기본 정보 확인
print(f"📊 데이터 크기: {df.shape[0]}건 × {df.shape[1]}컬럼\n")
print(f"📋 컬럼 목록: {df.columns.tolist()}\n")
print(f"📝 상위 5건 미리보기:")
df.head()

In [ ]:
# 데이터 구조 한눈에 보기
df.info()

## 2-2. 데이터 품질 점검 — 결측값 확인

In [ ]:
# 컬럼별 결측값 개수
missing = df.isnull().sum()
print("[ 컬럼별 결측값 ]")
print(missing[missing > 0])

# 결측 비율
print(f"\n[ 결측 비율(%) ]")
pct = (df.isnull().sum() / len(df) * 100).round(2)
print(pct[pct > 0])

In [ ]:
# 중복 행 확인
dup_count = df.duplicated().sum()
print(f"🔍 중복 행 수: {dup_count}건")

# 결측값이 있는 행 직접 보기
print(f"\n[ 정비일자 결측 행 ]")
print(df[df['정비일자'].isnull()])

## 2-3. 결측값 처리 전략

| 컬럼 | 결측 건수 | 전략 | 이유 |
|---|---|---|---|
| 정비일자 | 2건 | **행 삭제** | 날짜 없으면 시계열 분석 불가 |
| 고장유형 | 5건 | **'미확인' 대체** | 정비 기록은 존재, 가치 있음 |

In [ ]:
# ① 정비일자 결측 → 행 삭제
before = len(df)
df = df.dropna(subset=['정비일자']).reset_index(drop=True)
print(f"✅ 정비일자 결측 제거: {before} → {len(df)}건")

# ② 고장유형 결측 → '미확인'
df['고장유형'] = df['고장유형'].fillna('미확인')
print(f"✅ 고장유형 결측 처리: '미확인'으로 대체")

# ③ 최종 결측 확인
print(f"\n[ 처리 후 결측값 ]")
final_missing = df.isnull().sum()
print(final_missing[final_missing > 0] if final_missing.sum() > 0 else "✅ 결측값 없음")

## 2-4. 날짜 변환 + 파생변수 생성

In [ ]:
# 문자열 → datetime 변환
df['정비일자'] = pd.to_datetime(df['정비일자'])

# 파생변수 생성
df['월']   = df['정비일자'].dt.month
df['분기'] = df['정비일자'].dt.quarter
df['요일'] = df['정비일자'].dt.day_name()

# 한글 요일 매핑
weekday_kr = {'Monday':'월','Tuesday':'화','Wednesday':'수',
              'Thursday':'목','Friday':'금','Saturday':'토','Sunday':'일'}
df['요일_한글'] = df['요일'].map(weekday_kr)

print("[ 파생변수 생성 완료 ]")
df[['정비일자', '월', '분기', '요일_한글']].head()

---
### 🔥 실습문제 2
1. `df`의 **컬럼별 자료형(dtype)** 을 한 줄로 출력
2. `"결과"` 컬럼의 **고유값** 과 각 값의 **개수** 를 출력
3. `"담당부대"` 별로 **정비 기록 수** 를 내림차순으로 출력
4. 정비일자가 **2024-08-01 ~ 2024-08-31** 인 데이터의 **건수** 를 계산

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 컬럼별 자료형
print("[ 1. 컬럼별 자료형 ]")
print(df.dtypes)

# 2) 결과 고유값 & 개수
print("\n[ 2. 결과 분포 ]")
print(df['결과'].value_counts())

# 3) 담당부대별 기록 수
print("\n[ 3. 담당부대별 정비 건수 ]")
print(df['담당부대'].value_counts())

# 4) 8월 건수
aug = df[(df['정비일자'] >= '2024-08-01') & (df['정비일자'] <= '2024-08-31')]
print(f"\n[ 4. 2024년 8월 정비 건수: {len(aug)}건 ]")

---
# 📊 Chapter 3. [Step 3] 기술통계 분석

## 3-1. `describe()` — 수치형 전체 요약

In [ ]:
# 수치형 통계 요약
summary = df[['정비시간(h)', '정비비용(만원)']].describe().round(2)
print("[ 정비시간·비용 기술통계 ]")
print(summary)

In [ ]:
# 핵심 KPI 계산
total     = len(df)
avg_time  = df['정비시간(h)'].mean()
avg_cost  = df['정비비용(만원)'].mean()
down_rate = (df['결과'] == '가동불능').sum() / total * 100

print(f"📊 전체 정비 기록 수: {total}건")
print(f"⏱️ 평균 정비시간  : {avg_time:.2f} 시간")
print(f"💰 평균 정비비용  : {avg_cost:.0f} 만원")
print(f"⚠️ 가동불능 비율  : {down_rate:.1f}%")

## 3-2. 이상치 탐지 — IQR 방법

In [ ]:
# 정비시간 이상치 탐지
Q1, Q3 = df['정비시간(h)'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}h, Q3 = {Q3:.2f}h, IQR = {IQR:.2f}")
print(f"이상치 경계 : [{lower:.2f}, {upper:.2f}]")

# 이상치 추출
outliers = df[(df['정비시간(h)'] < lower) | (df['정비시간(h)'] > upper)]
print(f"\n⚠️ 이상치: {len(outliers)}건 (전체의 {len(outliers)/len(df)*100:.1f}%)")
print(outliers[['정비일자', '장비유형', '고장유형', '정비시간(h)', '정비비용(만원)']].head())

## 3-3. 범주형 변수 분석 — `value_counts()`

In [ ]:
# 장비유형별 정비 건수
print("[ 장비유형별 정비 건수 ]")
print(df['장비유형'].value_counts())

# 비율로 보기
print("\n[ 장비유형별 비율(%) ]")
print((df['장비유형'].value_counts(normalize=True) * 100).round(1))

In [ ]:
# 고장유형별 분포
print("[ 고장유형 분포 ]")
print(df['고장유형'].value_counts())

# 결과 분포
print("\n[ 정비 결과 분포 ]")
print(df['결과'].value_counts())

---
### 🔥 실습문제 3
1. **정비비용**의 평균, 중앙값, 표준편차, 최대값을 한 번에 출력
2. **정비비용** 이상치(IQR 방법)의 **건수** 와 **총 비용** 출력
3. 가장 고장이 많이 난 **장비번호 TOP 5** (`장비번호` 기준 `value_counts`)
4. `"완료"` 결과의 **비율** 이 몇 %인지 계산

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 정비비용 통계
cost = df['정비비용(만원)']
print(f"[ 1. 정비비용 통계 ]")
print(f"  평균    : {cost.mean():,.0f}만원")
print(f"  중앙값  : {cost.median():,.0f}만원")
print(f"  표준편차: {cost.std():,.0f}만원")
print(f"  최대    : {cost.max():,}만원")

# 2) 비용 이상치
Q1, Q3 = cost.quantile([0.25, 0.75])
IQR_c = Q3 - Q1
upper_c = Q3 + 1.5 * IQR_c
out_c = df[cost > upper_c]
print(f"\n[ 2. 비용 이상치 ]")
print(f"  건수    : {len(out_c)}건")
print(f"  총 비용 : {out_c['정비비용(만원)'].sum():,}만원")

# 3) 다회 고장 장비 TOP 5
print(f"\n[ 3. 반복 고장 장비 TOP 5 ]")
print(df['장비번호'].value_counts().head(5))

# 4) 완료 비율
pct_done = (df['결과'] == '완료').mean() * 100
print(f"\n[ 4. 완료율: {pct_done:.1f}% ]")

---
# 📈 Chapter 4. [Step 4] 단변량 분석 (Univariate Analysis)

> **한 개의 변수** 의 **분포와 특성** 을 파악하는 단계  
> → 히스토그램, 박스플롯이 주력 도구

## 4-1. 정비시간 분포 — 히스토그램

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['정비시간(h)'], bins=30,
         color='steelblue', edgecolor='white', alpha=0.8)

# 평균선과 중앙값선
mean_t   = df['정비시간(h)'].mean()
median_t = df['정비시간(h)'].median()
plt.axvline(mean_t,   color='red',      linestyle='--', linewidth=2, label=f'평균={mean_t:.1f}h')
plt.axvline(median_t, color='darkgreen',linestyle='--', linewidth=2, label=f'중앙값={median_t:.1f}h')

plt.title('정비시간 분포 (히스토그램)', fontsize=14, fontweight='bold')
plt.xlabel('정비시간 (h)'); plt.ylabel('빈도')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 분포 해석
skew = df['정비시간(h)'].skew()
print(f"왜도(Skewness): {skew:.3f}")
if skew > 1:
    print("→ 오른쪽 꼬리 분포 (우측 치우침). 평균보다 중앙값이 더 대표적 ✅")
elif skew < -1:
    print("→ 왼쪽 꼬리 분포 (좌측 치우침)")
else:
    print("→ 대체로 대칭 분포")

## 4-2. 월별 정비 건수 추이 — 선 그래프

In [ ]:
monthly = df.groupby('월').size().reindex(range(1, 13), fill_value=0)

plt.figure(figsize=(11, 5))
plt.plot(monthly.index, monthly.values,
         marker='o', color='coral', linewidth=2.5, markersize=10,
         markerfacecolor='white', markeredgewidth=2, markeredgecolor='coral')

# 최대값 강조
peak_month = monthly.idxmax()
peak_value = monthly.max()
plt.annotate(f'최고점\n{peak_value}건',
             xy=(peak_month, peak_value),
             xytext=(peak_month + 0.5, peak_value + 2),
             fontsize=11, fontweight='bold', color='red',
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# 각 지점에 값 표시
for m, v in monthly.items():
    plt.text(m, v + 0.5, str(v), ha='center', fontsize=9)

plt.title('2024년 월별 정비 건수 추이', fontsize=14, fontweight='bold')
plt.xlabel('월'); plt.ylabel('정비 건수')
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"💡 최다 정비월: {peak_month}월 ({peak_value}건)")
print(f"   → 하계 훈련(6~8월) 후 누적 손모 원인으로 추정")

## 4-3. 정비시간 박스플롯 — 이상치 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 전체 박스플롯
axes[0].boxplot(df['정비시간(h)'], patch_artist=True,
                boxprops=dict(facecolor='lightblue', edgecolor='navy'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('① 정비시간 전체 박스플롯', fontsize=13, fontweight='bold')
axes[0].set_ylabel('정비시간 (h)')

# 장비유형별 박스플롯
sns.boxplot(data=df, x='장비유형', y='정비시간(h)',
            palette='Set2', ax=axes[1])
axes[1].set_title('② 장비유형별 정비시간 분포', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

---
### 🔥 실습문제 4 - 단변량 분석
1. **정비비용** 분포의 **히스토그램** (bins=30) + 평균·중앙값선
2. **분기별(Q1~Q4)** 정비 건수 막대그래프
3. **장비유형별 정비비용** 박스플롯
4. **요일별** 정비 건수를 막대그래프로 시각화 (월~일 순)

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) 정비비용 히스토그램
axes[0,0].hist(df['정비비용(만원)'], bins=30,
               color='mediumpurple', edgecolor='white')
m = df['정비비용(만원)'].mean()
md_v = df['정비비용(만원)'].median()
axes[0,0].axvline(m, color='red', linestyle='--', label=f'평균={m:.0f}')
axes[0,0].axvline(md_v, color='green', linestyle='--', label=f'중앙값={md_v:.0f}')
axes[0,0].set_title('① 정비비용 분포', fontweight='bold')
axes[0,0].set_xlabel('비용(만원)'); axes[0,0].legend()

# 2) 분기별 정비 건수
q_cnt = df.groupby('분기').size()
bars = axes[0,1].bar(q_cnt.index, q_cnt.values,
                     color=['#FFB74D','#4DB6AC','#EF5350','#7986CB'], edgecolor='black')
for bar, v in zip(bars, q_cnt.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   str(v), ha='center', fontweight='bold')
axes[0,1].set_title('② 분기별 정비 건수', fontweight='bold')
axes[0,1].set_xticks([1,2,3,4])
axes[0,1].set_xticklabels(['Q1','Q2','Q3','Q4'])

# 3) 장비유형별 비용 박스플롯
sns.boxplot(data=df, x='장비유형', y='정비비용(만원)',
            palette='Set3', ax=axes[1,0])
axes[1,0].set_title('③ 장비유형별 정비비용', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=20)

# 4) 요일별 정비 건수
weekday_order = ['월','화','수','목','금','토','일']
wd_cnt = df['요일_한글'].value_counts().reindex(weekday_order, fill_value=0)
axes[1,1].bar(wd_cnt.index, wd_cnt.values, color='steelblue', edgecolor='black')
axes[1,1].set_title('④ 요일별 정비 건수', fontweight='bold')
axes[1,1].set_xlabel('요일')

plt.tight_layout()
plt.show()

---
# 🔗 Chapter 5. [Step 5] 다변량 분석 (Multivariate Analysis)

> **두 개 이상** 의 변수 간 **관계** 를 분석하는 단계  
> → groupby, 상관분석, heatmap이 주력 도구

## 5-1. 장비유형별 평균 정비비용 — 수평 막대그래프

In [ ]:
equip_cost = df.groupby('장비유형')['정비비용(만원)'].mean().sort_values()

plt.figure(figsize=(10, 5))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(equip_cost)))
bars = plt.barh(equip_cost.index, equip_cost.values,
                color=colors, edgecolor='black')

# 막대 끝에 값 표시
for bar, v in zip(bars, equip_cost.values):
    plt.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'{v:.0f}만원', va='center', fontweight='bold')

plt.title('장비유형별 평균 정비비용', fontsize=14, fontweight='bold')
plt.xlabel('평균 정비비용 (만원)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# 해석
max_equip = equip_cost.idxmax()
min_equip = equip_cost.idxmin()
ratio = equip_cost.max() / equip_cost.min()
print(f"💡 {max_equip} 비용이 {min_equip}의 {ratio:.1f}배")

## 5-2. 상관관계 분석 — `corr()` + 히트맵

In [ ]:
# 수치형 변수 간 상관계수
num_cols = ['정비시간(h)', '정비비용(만원)', '월']
corr = df[num_cols].corr()
print("[ 상관행렬 ]")
print(corr.round(2))

# 히트맵
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, square=True,
            cbar_kws={'label': '상관계수'})
plt.title('변수 간 상관관계 히트맵', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 산점도로 관계 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 정비시간 vs 비용 (장비유형별 색상)
sns.scatterplot(data=df, x='정비시간(h)', y='정비비용(만원)',
                hue='장비유형', palette='Set1', alpha=0.7, s=60,
                ax=axes[0])
axes[0].set_title('① 정비시간 vs 정비비용 (장비유형별)', fontweight='bold')
axes[0].grid(alpha=0.3)

# 회귀선 포함
sns.regplot(data=df, x='정비시간(h)', y='정비비용(만원)',
            scatter_kws={'alpha': 0.3, 's': 30},
            line_kws={'color': 'red', 'linewidth': 2},
            ax=axes[1])
axes[1].set_title('② 정비시간 vs 비용 회귀분석', fontweight='bold')

plt.tight_layout()
plt.show()

# 상관계수 p-value
r, p = stats.pearsonr(df['정비시간(h)'], df['정비비용(만원)'])
print(f"📊 정비시간 ↔ 정비비용: r={r:.3f}, p={p:.2e}")
print(f"   → {'**강한 양의 상관**' if r >= 0.7 else '약한 양의 상관'}")
print(f"   → 정비시간이 길수록 비용도 비례하여 증가")

## 5-3. 피벗 테이블 + 히트맵 — 월 × 장비유형

In [ ]:
# 월×장비유형 정비 건수 피벗
pivot_cnt = pd.pivot_table(
    df, values='장비번호', index='장비유형',
    columns='월', aggfunc='count', fill_value=0
)

plt.figure(figsize=(13, 4))
sns.heatmap(pivot_cnt, annot=True, fmt='d',
            cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': '정비 건수'})
plt.title('장비유형 × 월별 정비 건수 히트맵', fontsize=13, fontweight='bold')
plt.xlabel('월'); plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 월×장비유형 정비비용 합계 피벗
pivot_cost = pd.pivot_table(
    df, values='정비비용(만원)', index='장비유형',
    columns='월', aggfunc='sum', fill_value=0
)

plt.figure(figsize=(13, 4))
sns.heatmap(pivot_cost, annot=True, fmt='.0f',
            cmap='Blues', linewidths=0.5,
            cbar_kws={'label': '정비비용 합계(만원)'})
plt.title('장비유형 × 월별 정비비용 합계 히트맵', fontsize=13, fontweight='bold')
plt.xlabel('월'); plt.ylabel('')
plt.tight_layout()
plt.show()

# 최대 비용 월·장비
max_val = pivot_cost.max().max()
max_loc = pivot_cost.stack().idxmax()
print(f"\n💡 최대 비용 지점: {max_loc[1]}월 {max_loc[0]} ({max_val:,.0f}만원)")

---
### 🔥 실습문제 5 - 다변량 분석
1. **담당부대별** 평균 정비시간과 비용 (`groupby` + `agg`)
2. **고장유형별** 정비비용 박스플롯 (`sns.boxplot`)
3. **장비유형 × 결과** 교차표 (`crosstab`) — 각 장비유형별 가동불능 비율 계산
4. **담당부대 × 장비유형** 정비 건수 피벗 + 히트맵

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 담당부대별 평균 시간·비용
print("[ 1. 담당부대별 평균 ]")
unit_avg = df.groupby('담당부대').agg(
    평균시간 = ('정비시간(h)', 'mean'),
    평균비용 = ('정비비용(만원)', 'mean'),
    건수     = ('장비번호', 'count')
).round(1)
print(unit_avg)

# 2) 고장유형별 비용 박스플롯
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='고장유형', y='정비비용(만원)', palette='Set3')
plt.title('2. 고장유형별 정비비용 분포', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# 3) 장비유형 × 결과 교차표
print("\n[ 3. 장비유형 × 결과 교차표 ]")
ct = pd.crosstab(df['장비유형'], df['결과'])
print(ct)
# 가동불능 비율
ct['가동불능비율(%)'] = (ct['가동불능'] / ct.sum(axis=1) * 100).round(1)
print("\n[ 가동불능 비율 순위 ]")
print(ct['가동불능비율(%)'].sort_values(ascending=False))

# 4) 담당부대 × 장비유형 히트맵
plt.figure(figsize=(10, 4))
pv = pd.pivot_table(df, values='장비번호', index='담당부대',
                    columns='장비유형', aggfunc='count', fill_value=0)
sns.heatmap(pv, annot=True, fmt='d', cmap='YlGnBu', linewidths=0.5)
plt.title('4. 담당부대 × 장비유형 정비 건수', fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🎨 Chapter 6. 어떤 차트를 선택할까? — 차트 선택 가이드

## 6-1. 6가지 차트 유형 & 용도

| 차트 | 언제? | 예시 |
|---|---|---|
| 📊 **막대 차트** | 범주 간 크기 비교 | 장비별 건수 |
| 📈 **선 그래프** | 시간에 따른 변화·추세 | 월별 고장 추이 |
| 📉 **히스토그램** | 수치형 변수의 **분포** | 정비시간 분포 |
| 📦 **박스플롯** | **이상치 탐지** + 그룹 비교 | 장비별 비용 분포 |
| 🌡 **히트맵** | **상관관계** · 교차표 시각화 | 변수 간 상관 |
| 🔵 **산점도** | 두 변수 관계 · 군집 파악 | 시간 vs 비용 |

## ⚠️ 주의사항
- **파이 차트**는 범주 **6개 이상** 이면 오히려 읽기 어려움 → **막대 차트** 대체 권장
- **선 그래프**는 X축이 반드시 **시간 순서** 일 때만 사용!

In [ ]:
# 6가지 차트를 한 Figure에 비교
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('차트 선택 가이드 — 6가지 대표 차트', fontsize=16, fontweight='bold', y=1.00)

# ① 막대 차트 - 장비별 건수
eq_cnt = df['장비유형'].value_counts()
axes[0,0].bar(eq_cnt.index, eq_cnt.values,
              color='steelblue', edgecolor='black')
axes[0,0].set_title('① 막대 - 장비유형별 건수', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=20)

# ② 선 그래프 - 월별 추이
axes[0,1].plot(monthly.index, monthly.values, marker='o', color='coral', linewidth=2)
axes[0,1].set_title('② 선 - 월별 건수 추이', fontweight='bold')
axes[0,1].set_xticks(range(1,13))
axes[0,1].grid(alpha=0.3)

# ③ 히스토그램 - 시간 분포
axes[0,2].hist(df['정비시간(h)'], bins=25, color='mediumpurple', edgecolor='white')
axes[0,2].set_title('③ 히스토 - 정비시간 분포', fontweight='bold')

# ④ 박스플롯 - 장비별 시간
sns.boxplot(data=df, x='장비유형', y='정비시간(h)',
            palette='Set2', ax=axes[1,0])
axes[1,0].set_title('④ 박스 - 장비별 정비시간', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=20)

# ⑤ 히트맵 - 상관
sns.heatmap(df[['정비시간(h)','정비비용(만원)','월']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[1,1])
axes[1,1].set_title('⑤ 히트맵 - 상관관계', fontweight='bold')

# ⑥ 산점도 - 시간 vs 비용
sns.scatterplot(data=df, x='정비시간(h)', y='정비비용(만원)',
                hue='장비유형', palette='Set1', alpha=0.7,
                ax=axes[1,2], legend=False)
axes[1,2].set_title('⑥ 산점도 - 시간 vs 비용', fontweight='bold')

plt.tight_layout()
plt.show()

---
### 🔥 실습문제 6 - 차트 선택
다음 상황에서 가장 적절한 차트를 선택하세요.

1. 부대별 연간 정비비용 **총액** 을 한눈에 비교 → ?
2. **정비시간의 분포 모양**을 확인 → ?
3. 월별 가동불능 건수 **변화 추이** → ?
4. **정비시간 이상치** 존재 여부 확인 → ?
5. **정비시간과 비용의 관계** → ?
6. 장비유형×월 **정비건수 패턴** → ?

In [ ]:
# ✍️ 답을 작성하세요 (막대/선/히스토/박스/히트맵/산점도 중 선택)

answer_1 = ""
answer_2 = ""
answer_3 = ""
answer_4 = ""
answer_5 = ""
answer_6 = ""

print(f"1) 부대별 비용 총액 비교   → {answer_1}")
print(f"2) 정비시간 분포 모양      → {answer_2}")
print(f"3) 월별 가동불능 추이      → {answer_3}")
print(f"4) 정비시간 이상치 확인    → {answer_4}")
print(f"5) 시간과 비용 관계        → {answer_5}")
print(f"6) 장비×월 패턴            → {answer_6}")

**✅ 정답**

In [ ]:
answer_1 = "막대 차트"       # 범주 간 크기 비교
answer_2 = "히스토그램"        # 수치 분포
answer_3 = "선 그래프"         # 시간 추이
answer_4 = "박스플롯"          # 이상치 탐지
answer_5 = "산점도"            # 두 변수 관계
answer_6 = "히트맵"            # 교차 패턴

print(f"1) 부대별 비용 총액 비교   → {answer_1}   (범주 비교)")
print(f"2) 정비시간 분포 모양      → {answer_2}    (수치 분포)")
print(f"3) 월별 가동불능 추이      → {answer_3}    (시간 추이)")
print(f"4) 정비시간 이상치 확인    → {answer_4}     (이상치 탐지)")
print(f"5) 시간과 비용 관계        → {answer_5}       (두 변수 관계)")
print(f"6) 장비×월 패턴            → {answer_6}       (교차표)")

---
# 💡 Chapter 7. [Step 6] 인사이트 도출 및 보고

> **EDA의 최종 단계**: 분석 결과를 **의사결정 권고안** 으로 연결  
> 숫자·차트 → **행동 가능한 결론** 으로 변환

## 7-1. 인사이트 1 — 장비유형별 집중 관리 대상

In [ ]:
# 장비유형별 총 비용과 건수
equip_summary = df.groupby('장비유형').agg(
    정비건수 = ('장비번호', 'count'),
    총비용   = ('정비비용(만원)', 'sum'),
    평균시간 = ('정비시간(h)', 'mean'),
    가동불능 = ('결과', lambda x: (x == '가동불능').sum())
).round(1).sort_values('총비용', ascending=False)

equip_summary['비용비율(%)'] = (equip_summary['총비용'] / equip_summary['총비용'].sum() * 100).round(1)

print("🎯 장비유형별 정비 현황")
print(equip_summary)

# 최고 비용 장비
top_equip = equip_summary.index[0]
top_pct = equip_summary.iloc[0]['비용비율(%)']
print(f"\n💡 인사이트 1: {top_equip}이(가) 전체 정비비용의 {top_pct}% 차지")
print(f"   → 집중 관리 대상")

## 7-2. 인사이트 2 — 계절성 (월별 비용 패턴)

In [ ]:
# 월별 총 정비비용
monthly_cost = df.groupby('월')['정비비용(만원)'].sum()

plt.figure(figsize=(11, 5))
bars = plt.bar(monthly_cost.index, monthly_cost.values,
               color=['#90CAF9']*7 + ['#EF5350'] + ['#90CAF9']*4,
               edgecolor='black')

# 최대값 강조
peak = monthly_cost.idxmax()
peak_val = monthly_cost.max()
for i, (m, v) in enumerate(monthly_cost.items()):
    plt.text(m, v + 50, f'{v:,.0f}', ha='center',
             fontsize=9, fontweight='bold' if m == peak else 'normal',
             color='red' if m == peak else 'black')

plt.title('월별 정비비용 합계', fontsize=14, fontweight='bold')
plt.xlabel('월'); plt.ylabel('정비비용 합계 (만원)')
plt.xticks(range(1, 13))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 인사이트
Q3_cost = df[df['분기'] == 3]['정비비용(만원)'].sum()
total_cost = df['정비비용(만원)'].sum()
q3_pct = Q3_cost / total_cost * 100

print(f"💡 인사이트 2: {peak}월 비용 최고점 ({peak_val:,}만원)")
print(f"   → 3분기(7-9월, 하계 훈련기) 비용 비율: {q3_pct:.1f}%")
print(f"   → 7월 훈련 前 사전 정비 계획 수립 권고")

## 7-3. 인사이트 3 — 반복 고장 장비

In [ ]:
# 반복 고장 장비 TOP 5
repeat = df['장비번호'].value_counts().head(5)
print("🔄 반복 고장 장비 TOP 5")
print(repeat)

# 해당 장비의 고장 내역
top_unit = repeat.index[0]
top_history = df[df['장비번호'] == top_unit].sort_values('정비일자')
print(f"\n🔍 {top_unit}의 고장 내역:")
print(top_history[['정비일자', '고장유형', '정비시간(h)', '정비비용(만원)', '결과']].to_string(index=False))

total_cost_repeat = top_history['정비비용(만원)'].sum()
print(f"\n💡 인사이트 3: {top_unit}의 연간 정비 횟수 {len(top_history)}회, 누적 비용 {total_cost_repeat}만원")
print(f"   → 해당 장비 교체 또는 정밀 점검 필요")

## 7-4. 인사이트 4 — 가동불능 분석

In [ ]:
# 가동불능 사례 분석
downtime = df[df['결과'] == '가동불능']
print(f"⚠️ 가동불능 건수: {len(downtime)}건 (전체의 {len(downtime)/len(df)*100:.1f}%)")

# 고장유형별 가동불능 분포
dt_by_type = downtime['고장유형'].value_counts()
print(f"\n[ 가동불능 원인별 분포 ]")
print(dt_by_type)

# 시각화
plt.figure(figsize=(10, 4))
colors_dt = plt.cm.Reds(np.linspace(0.4, 0.8, len(dt_by_type)))
bars = plt.barh(dt_by_type.index, dt_by_type.values,
                color=colors_dt, edgecolor='black')
for bar, v in zip(bars, dt_by_type.values):
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{v}건', va='center', fontweight='bold')
plt.title('가동불능 원인 분포', fontsize=13, fontweight='bold')
plt.xlabel('건수')
plt.tight_layout()
plt.show()

main_cause = dt_by_type.idxmax()
print(f"\n💡 인사이트 4: 가동불능의 주요 원인은 '{main_cause}' ({dt_by_type.max()}건)")
print(f"   → 해당 부품의 여분 보유량 증가 검토")

---
# 🏆 Chapter 8. 종합 대시보드 + 최종 보고서

> 지금까지 분석한 내용을 **하나의 Figure** 와 **텍스트 보고서** 로 정리합니다.  
> 4일차 미니 프로젝트의 **최종 산출물**입니다.

## 8-1. 6종 차트 종합 대시보드

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('🎖️ 2024년 부대 장비 정비 EDA 종합 대시보드',
             fontsize=22, fontweight='bold', y=1.00)

# ① 장비유형별 정비 건수
eq_cnt = df['장비유형'].value_counts()
colors_eq = sns.color_palette('Set2', len(eq_cnt))
bars1 = axes[0,0].bar(eq_cnt.index, eq_cnt.values, color=colors_eq, edgecolor='black')
for bar, v in zip(bars1, eq_cnt.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   str(v), ha='center', fontweight='bold')
axes[0,0].set_title('① 장비유형별 정비 건수', fontsize=13, fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=15)

# ② 월별 정비비용 합계 추이
mc = df.groupby('월')['정비비용(만원)'].sum().reindex(range(1,13), fill_value=0)
axes[0,1].plot(mc.index, mc.values, marker='o', color='coral',
               linewidth=2.5, markersize=10, markerfacecolor='white', markeredgewidth=2)
peak_m = mc.idxmax()
axes[0,1].annotate(f'최고점 {peak_m}월',
                   xy=(peak_m, mc.max()), xytext=(peak_m-2, mc.max()+200),
                   fontsize=10, fontweight='bold', color='red',
                   arrowprops=dict(arrowstyle='->', color='red'))
axes[0,1].set_title('② 월별 정비비용 합계 추이', fontsize=13, fontweight='bold')
axes[0,1].set_xticks(range(1,13))
axes[0,1].grid(alpha=0.3)

# ③ 정비시간 분포 (히스토그램)
axes[0,2].hist(df['정비시간(h)'], bins=25, color='steelblue',
               edgecolor='white', density=True, alpha=0.7)
df['정비시간(h)'].plot(kind='kde', ax=axes[0,2], color='red', linewidth=2)
axes[0,2].set_title('③ 정비시간 분포 (히스토+KDE)', fontsize=13, fontweight='bold')
axes[0,2].set_xlabel('정비시간 (h)')

# ④ 장비유형별 비용 박스플롯
sns.boxplot(data=df, x='장비유형', y='정비비용(만원)',
            palette='Pastel2', ax=axes[1,0])
axes[1,0].set_title('④ 장비유형별 정비비용 (이상치)', fontsize=13, fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=15)

# ⑤ 시간 vs 비용 산점도
sns.scatterplot(data=df, x='정비시간(h)', y='정비비용(만원)',
                hue='장비유형', palette='Set1', alpha=0.6, s=50,
                ax=axes[1,1])
axes[1,1].set_title('⑤ 정비시간 vs 정비비용', fontsize=13, fontweight='bold')
axes[1,1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# ⑥ 결과 분포 파이 차트
rc = df['결과'].value_counts()
axes[1,2].pie(rc, labels=rc.index, autopct='%1.1f%%',
              startangle=90, colors=['#66BB6A','#FFCA28','#EF5350'],
              shadow=True, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1,2].set_title('⑥ 정비 결과 분포', fontsize=13, fontweight='bold')

plt.tight_layout()

# PNG로 저장
plt.savefig('/content/EDA_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("✅ /content/EDA_dashboard.png 저장 완료")
plt.show()

## 8-2. 최종 보고서 자동 생성

In [ ]:
print("=" * 60)
print("📋 2024년 부대 장비 정비 EDA 분석 보고서")
print("=" * 60)

# ── 1. 개요 ──
print(f"\n【 1. 분석 개요 】")
print(f"  • 분석 기간    : 2024-01 ~ 2024-12 (12개월)")
print(f"  • 총 정비 건수 : {len(df):,}건")
print(f"  • 총 정비비용  : {df['정비비용(만원)'].sum():,}만원")
print(f"  • 평균 정비시간: {df['정비시간(h)'].mean():.2f}시간")
print(f"  • 가동불능율   : {(df['결과']=='가동불능').mean()*100:.1f}%")

# ── 2. 주요 발견 ──
print(f"\n【 2. 주요 발견 】")

# 발견 1: 고비용 장비
top_eq = df.groupby('장비유형')['정비비용(만원)'].sum().sort_values(ascending=False)
print(f"  [발견 1] 최고비용 장비: {top_eq.index[0]}")
print(f"           → 총 {top_eq.iloc[0]:,}만원 (전체의 {top_eq.iloc[0]/top_eq.sum()*100:.1f}%)")

# 발견 2: 정비 집중월
peak_m = df.groupby('월').size().idxmax()
peak_cnt = df.groupby('월').size().max()
print(f"\n  [발견 2] 최다 정비월: {peak_m}월 ({peak_cnt}건)")
print(f"           → 하계 훈련(6~8월) 직후 누적 손모 원인 추정")

# 발견 3: 강한 상관
r, p = stats.pearsonr(df['정비시간(h)'], df['정비비용(만원)'])
print(f"\n  [발견 3] 정비시간 ↔ 비용 상관계수: r={r:.3f} (강한 양의 상관)")
print(f"           → 시간 단축 = 비용 절감 효과")

# 발견 4: 반복 고장
repeat = df['장비번호'].value_counts()
repeat_3plus = (repeat >= 3).sum()
print(f"\n  [발견 4] 연 3회 이상 정비 장비: {repeat_3plus}대")
print(f"           → 교체 또는 정밀 점검 대상")

# ── 3. 권고사항 ──
print(f"\n【 3. 의사결정 권고안 】")
print(f"  ① {top_eq.index[0]} 전용 예방정비 프로그램 수립")
print(f"     → 월 1회 정기 점검, 부품 사전 확보")
print(f"  ② 7월 이전 사전 정비 계획 수립")
print(f"     → 하계 훈련 前 주요 장비 종합 점검")
print(f"  ③ 반복 고장 장비 {repeat_3plus}대 긴급 점검")
print(f"     → 노후 부품 교체 또는 장비 교체")
print(f"  ④ 가동불능 주요 원인 부품 여분 확보")

print("\n" + "=" * 60)
print(f"  📅 보고일: {pd.Timestamp.today().strftime('%Y-%m-%d')}")
print(f"  📂 대시보드: /content/EDA_dashboard.png")
print("=" * 60)

## 8-3. 분석 결과를 CSV로 저장

In [ ]:
# 핵심 요약 표 저장
equip_final = df.groupby('장비유형').agg(
    정비건수 = ('장비번호', 'count'),
    총비용   = ('정비비용(만원)', 'sum'),
    평균시간 = ('정비시간(h)', 'mean'),
    가동불능 = ('결과', lambda x: (x == '가동불능').sum())
).round(2).sort_values('총비용', ascending=False)

equip_final.to_csv('/content/EDA_summary.csv', encoding='utf-8-sig')

# 월별 추이 저장
monthly_summary = df.groupby('월').agg(
    건수 = ('장비번호', 'count'),
    총비용 = ('정비비용(만원)', 'sum'),
    평균시간 = ('정비시간(h)', 'mean')
).round(2)
monthly_summary.to_csv('/content/EDA_monthly.csv', encoding='utf-8-sig')

print("✅ 저장 완료")
print("  - /content/EDA_summary.csv  (장비유형 요약)")
print("  - /content/EDA_monthly.csv  (월별 추이)")
print("  - /content/EDA_dashboard.png (종합 대시보드)")

---
# 🚀 Chapter 9. 스스로 해보는 미니 프로젝트

> 지금까지 배운 6단계 EDA를 **다른 주제** 로 직접 실행해 보세요.  
> 막히면 **Vibe Coding(ChatGPT/Claude)** 을 활용해 보세요!

## 🎯 미션: 담당부대 관점의 EDA
지금까지는 **장비유형** 관점에서 분석했습니다. 이제 **담당부대** 관점에서 다시 분석하세요.

### 요구사항
1. **담당부대별 정비 건수와 총 비용** 순위 (막대그래프)
2. **부대 × 장비유형 피벗 테이블** (정비 건수) + 히트맵
3. **부대별 월 추이** (4개 부대의 월별 정비 건수 선 그래프)
4. **부대별 가동불능률** 계산 및 비교
5. **한 줄 인사이트** 3개 이상 도출 (print 문으로)

In [ ]:
# ✍️ 여기에 본인만의 EDA 분석을 작성하세요
# 힌트: df는 이미 로드되어 있습니다. 아래 3-4개의 그룹화/피벗/시각화 조합으로 시작하세요.

# 1) 담당부대별 정비 건수와 총 비용

# 2) 부대 × 장비유형 피벗

# 3) 부대별 월 추이

# 4) 부대별 가동불능률

# 5) 인사이트 출력


**✅ 샘플 솔루션**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('📊 담당부대 관점 EDA', fontsize=18, fontweight='bold', y=1.00)

# 1) 부대별 정비 건수와 총 비용
unit_stat = df.groupby('담당부대').agg(
    건수 = ('장비번호', 'count'),
    총비용 = ('정비비용(만원)', 'sum')
).sort_values('총비용', ascending=False)

ax = axes[0,0]
ax2 = ax.twinx()
x = range(len(unit_stat))
ax.bar(x, unit_stat['건수'], color='steelblue', alpha=0.7, label='건수')
ax2.plot(x, unit_stat['총비용'], marker='o', color='red', linewidth=2, markersize=10, label='총비용')
ax.set_xticks(x); ax.set_xticklabels(unit_stat.index)
ax.set_ylabel('정비 건수', color='steelblue')
ax2.set_ylabel('총 비용 (만원)', color='red')
ax.set_title('① 부대별 정비 건수·총비용', fontweight='bold')

# 2) 부대 × 장비유형 피벗 히트맵
pv = pd.pivot_table(df, values='장비번호', index='담당부대',
                    columns='장비유형', aggfunc='count', fill_value=0)
sns.heatmap(pv, annot=True, fmt='d', cmap='YlGnBu',
            linewidths=0.5, ax=axes[0,1])
axes[0,1].set_title('② 부대 × 장비유형 정비 건수', fontweight='bold')

# 3) 부대별 월 추이
for unit in df['담당부대'].unique():
    unit_monthly = df[df['담당부대']==unit].groupby('월').size().reindex(range(1,13), fill_value=0)
    axes[1,0].plot(unit_monthly.index, unit_monthly.values,
                   marker='o', linewidth=1.8, label=unit)
axes[1,0].set_title('③ 부대별 월별 정비 건수', fontweight='bold')
axes[1,0].set_xticks(range(1,13))
axes[1,0].set_xlabel('월'); axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# 4) 부대별 가동불능률
down_rate = df.groupby('담당부대').apply(
    lambda g: (g['결과']=='가동불능').mean() * 100
).sort_values(ascending=False).round(2)
bars = axes[1,1].bar(down_rate.index, down_rate.values,
                     color=['#EF5350','#FFA726','#FFEE58','#81C784'], edgecolor='black')
for bar, v in zip(bars, down_rate.values):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{v:.1f}%', ha='center', fontweight='bold')
axes[1,1].set_title('④ 부대별 가동불능률', fontweight='bold')
axes[1,1].set_ylabel('가동불능률 (%)')

plt.tight_layout()
plt.show()

# 5) 인사이트
print("\n" + "=" * 50)
print("💡 담당부대 관점 인사이트")
print("=" * 50)
worst_unit = unit_stat.index[0]
print(f"① 정비비용 최다 부대: {worst_unit} ({unit_stat.iloc[0]['총비용']:,}만원)")
print(f"   → 장비 노후화 또는 사용 강도 조사 필요")

high_down = down_rate.idxmax()
print(f"\n② 가동불능률 최다: {high_down} ({down_rate.max():.1f}%)")
print(f"   → 장비 관리 체계 개선 권고")

busy_month = df.groupby('월').size().idxmax()
print(f"\n③ 전체 {busy_month}월 정비 집중")
print(f"   → 모든 부대에 사전 정비 가이드 배포")

---
# 🎓 4일차 & 30시간 전체 교육 수료 정리

## ✅ 4일차 배운 내용
| 단계 | 도구 | 결과물 |
|---|---|---|
| **1. 데이터 로드** | `read_csv`, `info`, `shape` | 데이터 구조 파악 |
| **2. 품질 점검** | `isnull`, `dropna`, `fillna` | 깨끗한 데이터 |
| **3. 기술통계** | `describe`, `value_counts`, IQR | 분포 요약 |
| **4. 단변량 분석** | 히스토그램, 박스플롯 | 개별 변수 특성 |
| **5. 다변량 분석** | `groupby`, `corr`, 히트맵 | 변수 간 관계 |
| **6. 인사이트** | 자동 보고서 + 대시보드 | **의사결정 권고** |

## 🏆 30시간 전체 수료 한눈에 보기
| 일차 | 주제 | 핵심 기술 |
|---|---|---|
| **1일차** | 파이썬 기초 & Colab | 변수·조건문·라이브러리·Vibe Coding |
| **2일차** | 데이터 수집 & 전처리 | pandas, 결측치·이상치·정규화 |
| **3일차** | 데이터 분석 & 가공 | groupby·pivot·corr·시각화 |
| **4일차** | EDA 미니 프로젝트 | 종합 분석 파이프라인 |

## 💡 John Tukey의 마지막 메시지
> **"데이터를 먼저 이해하고, 그 다음 분석하라."**  
> EDA는 분석의 시작이자, **의사결정의 근거**를 만드는 과정입니다.

## 🚀 수료 후 자기계발 로드맵
| Lv | 단계 | 추천 |
|---|---|---|
| 1 | **기초 유지** | 오늘 코드를 본인 부대 데이터에 적용 |
| 2 | **통계 강화** | 통계학 기초, 가설검정 |
| 3 | **ML 입문** | scikit-learn, 분류·회귀 모델 |
| 4 | **딥러닝** | PyTorch, TensorFlow |
| 5 | **AI 자동화** | API 활용, 업무 자동화 파이프라인 |

### 📚 추천 학습 자원
- **KMOOC** — 무료 온라인 강좌
- **유튜브 "모두의 AI"** — 한국어 AI 튜토리얼
- **Anthropic Claude** — 코딩·분석 AI 어시스턴트
- **Kaggle** — 실전 데이터 분석 프로젝트

---

### ⚠️ AI 활용 시 반드시 지켜야 할 수칙
- 🚫 **군사기밀·개인정보 입력 금지**
- 🔍 **AI 결과 반드시 검증** (맹신 금지)
- 📄 **저작권 및 보안 유의**
- ✅ **최종 책임은 사용자** — AI는 도구

---

## 🎖️ 수료를 축하합니다!

> **"스마트 강군 육성, 국방 AX(AI 전환)에 만전을 기하겠습니다"**

오늘 배운 EDA 기술을 본인 부대 실무 데이터에 적용하여  
**데이터 기반 의사결정 문화**를 만들어가세요!

---

*교육기관: (사)한국오픈소스협회 | 문의: 02-6012-7414 / kmil@osskorea.org*